In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, glob
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

BASE_DIR = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output"

GRID_PATH = os.path.join(BASE_DIR, "taxi_zone_lookup_grid.csv")
grid = pd.read_csv(GRID_PATH)

# Quick overview structured data from Raw data

In [4]:
print(grid.head())
print("Num zones:", len(grid))
print("Grid_X range:", grid["Grid_X"].min(), grid["Grid_X"].max())
print("Grid_Y range:", grid["Grid_Y"].min(), grid["Grid_Y"].max())

   LocationID  Grid_X  Grid_Y
0           1       1       8
1           2       7       4
2           3       7      18
3           4       4      10
4           5       0       1
Num zones: 263
Grid_X range: 0 9
Grid_Y range: 0 19


In [13]:
print("Nums loca:", grid["LocationID"].nunique())

Nums loca: 263


### Đọc thử dữ liệu sau khi clean: Volumn và Flow

In [6]:
DATASET = "yellow"
YEAR = "2024"
MONTH = "01"

VOL_PATH = os.path.join(BASE_DIR, DATASET + "_data", DATASET + "_" + YEAR, f"{MONTH}_volume.csv")
FLOW_PATH = os.path.join(BASE_DIR, DATASET + "_data", DATASET + "_" + YEAR, f"{MONTH}_flow.parquet")

VOL: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2024/01_volume.csv True
FLOW: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2024/01_flow.parquet True


In [7]:
vol = pd.read_csv(VOL_PATH)
vol["time_bin"] = pd.to_datetime(vol["time_bin"], errors="coerce")

print(vol.head())

             time_bin  locationid  start_volume  end_volume
0 2002-12-31 22:30:00         170             1           1
1 2023-12-31 23:30:00          68             1           1
2 2023-12-31 23:30:00          90             1           0
3 2023-12-31 23:30:00         137             0           1
4 2023-12-31 23:30:00         138             1           0


In [8]:
print("Rows:", len(vol))
print("Unique time bins:", vol["time_bin"].nunique())
print("Unique locations:", vol["locationid"].nunique())
print(vol[["start_volume","end_volume"]].describe())

Rows: 193362
Unique time bins: 1491
Unique locations: 260
        start_volume     end_volume
count  193362.000000  193362.000000
mean       13.949602      13.949602
std        30.457559      25.529075
min         0.000000       0.000000
25%         0.000000       1.000000
50%         1.000000       3.000000
75%        10.000000      14.000000
max       405.000000     318.000000


In [15]:
# số location trung bình mỗi time_bin
cnt_per_t = vol.groupby("time_bin")["locationid"].nunique()
print(cnt_per_t.describe())

count    1491.000000
mean      129.686117
std        25.895986
min         1.000000
25%       121.000000
50%       138.000000
75%       147.000000
max       166.000000
Name: locationid, dtype: float64


In [16]:
pf = pq.ParquetFile(FLOW_PATH)
print("Num row groups:", pf.num_row_groups)
print("Schema:")
print(pf.schema)

Num row groups: 2
Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 time_bin (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 pulocationid;
  optional int32 field_id=-1 dolocationid;
  optional int64 field_id=-1 flow_count;
}



In [20]:
sample = pf.read_row_group(0).to_pandas()
print(sample.head(10))
print("Sample rows:", len(sample))

             time_bin  pulocationid  dolocationid  flow_count
0 2002-12-31 22:30:00           170           170           1
1 2023-12-31 23:30:00            68           137           1
2 2023-12-31 23:30:00            90            68           1
3 2023-12-31 23:30:00           138           217           1
4 2023-12-31 23:30:00           144           211           1
5 2023-12-31 23:30:00           161           170           1
6 2023-12-31 23:30:00           163           237           1
7 2023-12-31 23:30:00           229           244           1
8 2023-12-31 23:30:00           234           237           1
9 2023-12-31 23:30:00           236           142           1
Sample rows: 1048576


In [23]:
sample.groupby(["time_bin"]).size().head(10)

,0
time_bin,
2002-12-31 22:30:00,1
2023-12-31 23:30:00,10
2024-01-01 00:00:00,1261
2024-01-01 00:30:00,1486
2024-01-01 01:00:00,1417
2024-01-01 01:30:00,1499
2024-01-01 02:00:00,1543
2024-01-01 02:30:00,1395
2024-01-01 03:00:00,1256


# Check every year from 2018 - 2024 to choose which months in which year to use for training models

In [29]:
grid = pd.read_csv(GRID_PATH)
TOTAL_ZONES = grid["LocationID"].nunique()

print("TOTAL_ZONES:", TOTAL_ZONES)
print("Grid W:", grid["Grid_X"].max()+1, "H:", grid["Grid_Y"].max()+1)

TOTAL_ZONES: 263
Grid W: 10 H: 20


In [30]:
def expected_bins(year: int, month: int, interval_min=30):
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    return int(((end - start).total_seconds() / 60) / interval_min)

def analyze_volume_csv(vol_path: str, year: int, month: int):
    vol = pd.read_csv(vol_path)
    vol["time_bin"] = pd.to_datetime(vol["time_bin"], errors="coerce")
    vol = vol.dropna(subset=["time_bin"])

    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)
    in_month = vol[(vol["time_bin"] >= start) & (vol["time_bin"] < end)].copy()
    out_month = len(vol) - len(in_month)

    if len(in_month) == 0:
        return {
            "rows_vol": len(vol),
            "rows_in_month": 0,
            "rows_out_month": out_month,
            "n_time_bins": 0,
            "expected_bins": expected_bins(year, month),
            "bin_coverage": 0.0,
            "unique_locations": 0,
            "loc_coverage": 0.0,
            "median_loc_per_bin": 0.0,
            "mean_start": np.nan,
            "p75_start": np.nan,
            "max_start": np.nan,
            "ratio_ge10": np.nan
        }

    in_month["locationid"] = pd.to_numeric(in_month["locationid"], errors="coerce").astype("Int64")
    in_month["start_volume"] = pd.to_numeric(in_month["start_volume"], errors="coerce").fillna(0)
    in_month = in_month.dropna(subset=["locationid"])
    in_month["locationid"] = in_month["locationid"].astype(int)

    n_bins = in_month["time_bin"].nunique()
    exp_bins = expected_bins(year, month)
    bin_cov = n_bins / exp_bins

    uniq_loc = in_month["locationid"].nunique()
    loc_cov = uniq_loc / TOTAL_ZONES

    loc_per_bin = in_month.groupby("time_bin")["locationid"].nunique()
    median_loc_per_bin = float(loc_per_bin.median())

    sv = in_month["start_volume"].astype(float)
    ratio_ge10 = float((sv >= 10).mean())

    return {
        "rows_vol": len(vol),
        "rows_in_month": len(in_month),
        "rows_out_month": out_month,
        "n_time_bins": int(n_bins),
        "expected_bins": int(exp_bins),
        "bin_coverage": float(bin_cov),
        "unique_locations": int(uniq_loc),
        "loc_coverage": float(loc_cov),
        "median_loc_per_bin": median_loc_per_bin,
        "mean_start": float(sv.mean()),
        "p75_start": float(sv.quantile(0.75)),
        "max_start": float(sv.max()),
        "ratio_ge10": ratio_ge10
    }

In [31]:
import pyarrow as pa
import pyarrow.compute as pc

def analyze_flow_parquet(flow_path: str, year: int, month: int):
    pf = pq.ParquetFile(flow_path)

    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    total_rows = 0
    in_rows = 0
    total_flow = 0

    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(rg, columns=["time_bin", "flow_count"])
        total_rows += tbl.num_rows

        t = tbl["time_bin"]
        mask = pc.and_(pc.greater_equal(t, pa.scalar(start.to_datetime64())),
                       pc.less(t, pa.scalar(end.to_datetime64())))
        tbl_m = tbl.filter(mask)
        in_rows += tbl_m.num_rows

        if tbl_m.num_rows > 0:
            total_flow += int(pc.sum(tbl_m["flow_count"]).as_py())

    return {
        "rows_flow": int(total_rows),
        "rows_flow_in_month": int(in_rows),
        "sum_flow_in_month": int(total_flow),
        "flow_in_month_ratio": float(in_rows / total_rows) if total_rows else 0.0
    }

In [32]:
def scan_dataset(dataset_name: str, years=range(2018, 2024)):
    rows = []
    for y in years:
        year_dir = os.path.join(BASE_DIR, f"{dataset_name}_data", f"{dataset_name}_{y}")
        if not os.path.isdir(year_dir):
            continue

        for m in range(1, 13):
            mm = f"{m:02d}"
            vol_path = os.path.join(year_dir, f"{mm}_volume.csv")
            flow_path = os.path.join(year_dir, f"{mm}_flow.parquet")

            if not os.path.exists(vol_path):
                continue

            rec = {"dataset": dataset_name, "year": y, "month": mm}
            rec.update(analyze_volume_csv(vol_path, y, m))

            if os.path.exists(flow_path):
                rec.update(analyze_flow_parquet(flow_path, y, m))
            else:
                rec.update({"rows_flow": np.nan, "rows_flow_in_month": np.nan, "sum_flow_in_month": np.nan, "flow_in_month_ratio": np.nan})

            rows.append(rec)

    return pd.DataFrame(rows)

In [33]:
df_yellow = scan_dataset("yellow")

print("yellow months scanned:", len(df_yellow))

yellow months scanned: 72


### Prioritize bin_coverage, loc_coverage, ratio_ge10, median_loc_per_bin/TOTAL_ZONES and put heavy penalty when rows_out_month is big

In [34]:
df_yellow.head(20)

,dataset,year,month,rows_vol,rows_in_month,rows_out_month,n_time_bins,expected_bins,bin_coverage,unique_locations,loc_coverage,median_loc_per_bin,mean_start,p75_start,max_start,ratio_ge10,rows_flow,rows_flow_in_month,sum_flow_in_month,flow_in_month_ratio
0,yellow,2018,01,244016,243588,428,1488,1488,1.000000,259,0.984791,165.0,34.808796,23.0,914.0,0.320640,2762997,2762672,8479005,0.999882
1,yellow,2018,02,223173,222169,1004,1344,1344,1.000000,260,0.988593,166.0,37.081384,25.0,814.0,0.325712,2621369,2620685,8238334,0.999739
2,yellow,2018,03,255689,253817,1872,1487,1488,0.999328,261,0.992395,173.0,35.960196,24.0,906.0,0.321688,2957003,2955711,9127309,0.999563
3,yellow,2018,04,250935,250416,519,1440,1440,1.000000,259,0.984791,177.0,36.000599,22.0,857.0,0.313578,2902563,2902101,9015126,0.999841
4,yellow,2018,05,260537,260203,334,1488,1488,1.000000,261,0.992395,178.0,34.406148,21.0,726.0,0.313190,2989293,2988988,8952583,0.999898
5,yellow,2018,06,252774,252252,522,1440,1440,1.000000,259,0.984791,178.0,33.481701,21.0,720.0,0.313230,2889571,2889070,8445826,0.999827
6,yellow,2018,07,255905,255336,569,1488,1488,1.000000,260,0.988593,174.0,29.772829,19.0,650.0,0.302660,2763501,2763008,7602075,0.999822
7,yellow,2018,08,260721,260233,488,1488,1488,1.000000,260,0.988593,178.0,29.232530,18.0,649.0,0.296830,2773669,2773251,7607269,0.999849
8,yellow,2018,09,250886,250357,529,1440,1440,1.000000,261,0.992395,178.0,31.110762,19.0,718.0,0.303071,2751917,2751480,7788797,0.999841
9,yellow,2018,10,261019,260546,473,1488,1488,1.000000,261,0.992395,180.0,32.588065,19.0,809.0,0.300304,2870363,2870017,8490690,0.999879


In [35]:
df_green = scan_dataset("green")

print("green months scanned:", len(df_green))

green months scanned: 72


In [36]:
df_green.head(20)

,dataset,year,month,rows_vol,rows_in_month,rows_out_month,n_time_bins,expected_bins,bin_coverage,unique_locations,loc_coverage,median_loc_per_bin,mean_start,p75_start,max_start,ratio_ge10,rows_flow,rows_flow_in_month,sum_flow_in_month,flow_in_month_ratio
0,green,2018,01,211462,211228,234,1488,1488,1.000000,258,0.980989,157.5,3.684379,3.0,131.0,0.107192,506836,506695,778244,0.999722
1,green,2018,02,199346,198970,376,1344,1344,1.000000,256,0.973384,164.0,3.799774,3.0,129.0,0.108303,493108,492873,756041,0.999523
2,green,2018,03,223536,223104,432,1487,1488,0.999328,259,0.984791,166.0,3.679867,3.0,112.0,0.104781,545983,545701,820993,0.999484
3,green,2018,04,219734,219477,257,1440,1440,1.000000,257,0.977186,169.0,3.573805,3.0,109.0,0.102056,535377,535224,784368,0.999714
4,green,2018,05,227389,227277,112,1488,1488,1.000000,257,0.977186,168.0,3.432697,3.0,101.0,0.098461,547036,546964,780173,0.999868
5,green,2018,06,220616,220471,145,1440,1440,1.000000,258,0.980989,169.0,3.283089,3.0,95.0,0.094534,516679,516580,723826,0.999808
6,green,2018,07,219118,218997,121,1488,1488,1.000000,258,0.980989,163.0,3.060220,2.0,100.0,0.089083,484566,484496,670179,0.999856
7,green,2018,08,218613,218462,151,1488,1488,1.000000,258,0.980989,164.0,2.987407,2.0,89.0,0.086651,477549,477452,652635,0.999797
8,green,2018,09,214082,213839,243,1440,1440,1.000000,258,0.980989,166.0,3.056388,3.0,100.0,0.087327,477484,477345,653575,0.999709
9,green,2018,10,224335,224181,154,1488,1488,1.000000,258,0.980989,169.0,3.106030,3.0,94.0,0.087010,508664,508568,696313,0.999811


In [50]:
def add_quality_score(df):
    df = df.copy()
    df["out_ratio"] = (df["rows_out_month"] / df["rows_vol"]).fillna(1.0)

    df["score"] = (
        0.40 * df["bin_coverage"].clip(0,1) +
        0.25 * df["loc_coverage"].clip(0,1) +
        0.25 * df["ratio_ge10"].clip(0,1) +
        0.10 * (df["median_loc_per_bin"] / TOTAL_ZONES).clip(0,1)
        - 0.80 * df["out_ratio"].clip(0,1)
    )
    return df

In [52]:
yellow_q = add_quality_score(df_yellow)

yellow_q.sort_values("score", ascending=False).head(30)[
    ["dataset","year","month","score","bin_coverage","loc_coverage","ratio_ge10","rows_out_month","sum_flow_in_month"]
]

,dataset,year,month,score,bin_coverage,loc_coverage,ratio_ge10,rows_out_month,sum_flow_in_month
4,yellow,2018,05,0.793051,1.000000,0.992395,0.313190,334,8952583
5,yellow,2018,06,0.790534,1.000000,0.984791,0.313230,522,8445826
3,yellow,2018,04,0.790238,1.000000,0.984791,0.313578,519,9015126
9,yellow,2018,10,0.790166,1.000000,0.992395,0.300304,473,8490690
8,yellow,2018,09,0.789860,1.000000,0.992395,0.303071,529,7788797
10,yellow,2018,11,0.788726,1.000000,0.992395,0.295045,377,7816653
2,yellow,2018,03,0.788175,0.999328,0.992395,0.321688,1872,9127309
1,yellow,2018,02,0.788095,1.000000,0.988593,0.325712,1004,8238334
0,yellow,2018,01,0.787692,1.000000,0.984791,0.320640,428,8479005
7,yellow,2018,08,0.787539,1.000000,0.988593,0.296830,488,7607269


In [53]:
green_q  = add_quality_score(df_green)

green_q.sort_values("score", ascending=False).head(30)[
    ["dataset","year","month","score","bin_coverage","loc_coverage","ratio_ge10","rows_out_month","sum_flow_in_month"]
]

,dataset,year,month,score,bin_coverage,loc_coverage,ratio_ge10,rows_out_month,sum_flow_in_month
2,green,2018,03,0.733696,0.999328,0.984791,0.104781,432,820993
3,green,2018,04,0.733134,1.000000,0.977186,0.102056,257,784368
5,green,2018,06,0.732613,1.000000,0.980989,0.094534,145,723826
4,green,2018,05,0.732396,1.000000,0.977186,0.098461,112,780173
1,green,2018,02,0.731270,1.000000,0.973384,0.108303,376,756041
0,green,2018,01,0.731046,1.000000,0.980989,0.107192,234,778244
9,green,2018,10,0.730709,1.000000,0.980989,0.087010,154,696313
8,green,2018,09,0.729289,1.000000,0.980989,0.087327,243,653575
11,green,2018,12,0.729101,1.000000,0.984791,0.083336,184,669173
6,green,2018,07,0.729053,1.000000,0.980989,0.089083,121,670179


## => 2018's data seems to be good to use for its well match for our problems and also for the 2 models demanding input